In [1]:
import os
from getpass import getpass

# 1. Groq API Key
print("Enter your Groq API Key:")
os.environ["GROQ_API_KEY"] = getpass()

# 2. LangSmith Tracing Setup 
print("Enter your LangSmith API Key:")
os.environ["LANGCHAIN_API_KEY"] = getpass()
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "groq-resume-screening"

Enter your Groq API Key:


 ········


Enter your LangSmith API Key:


 ········


In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq

# --- 1. Prompts ---
extract_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""You are an expert HR assistant. Extract the following details from the provided resume.
Do NOT assume or hallucinate skills that are not explicitly present in the resume.

Extract:
1. Skills
2. Years of Experience
3. Tools/Technologies

Resume:
{resume}

Extracted Output:
"""
)

eval_prompt = PromptTemplate(
    input_variables=["job_description", "extracted_profile"],
    template="""You are a technical recruiter. Compare the candidate's extracted profile against the job description.

Job Description:
{job_description}

Extracted Candidate Profile:
{extracted_profile}

Evaluate the candidate and provide the following strictly in this structure:
MATCHING SUMMARY: (Briefly describe how well the skills match)
FIT SCORE: (Assign a score from 0 to 100)
EXPLANATION: (Detailed reasoning for why this score was assigned)
"""
)

# --- 2. LLM Initialization ---
llm = ChatGroq(
    model="llama-3.1-8b-instant", # Updated from llama3-8b-8192
    temperature=0.1,
    max_tokens=500
)

# --- 3. Pipeline Construction ---
extraction_chain = extract_prompt | llm

pipeline = (
    {
        "extracted_profile": {"resume": lambda x: x["resume"]} | extraction_chain,
        "job_description": lambda x: x["job_description"]
    }
    | eval_prompt
    | llm
)

In [5]:
# --- 1. Define Data ---
job_description = """
Data Scientist required. 
Must have 4+ years of experience in Data Science.
Skills required: Python, Machine Learning, Deep Learning, NLP.
Tools: LangChain, TensorFlow, PyTorch, Jupyter, Git.
"""

resumes = {
    "Strong Candidate": """
    Jane Doe
    Experience: 5 years as a Senior Data Scientist.
    Skills: Python, Machine Learning, Deep Learning, NLP, Generative AI.
    Tools: LangChain, PyTorch, TensorFlow, Git, Jupyter Notebooks, Docker.
    """,
    
    "Average Candidate": """
    John Smith
    Experience: 2 years as a Data Analyst.
    Skills: Python, Data Visualization, basic Machine Learning.
    Tools: Pandas, Scikit-learn, SQL, Jupyter.
    """,
    
    "Weak Candidate": """
    Alice Johnson
    Experience: 1 year as a Frontend Developer.
    Skills: HTML, CSS, JavaScript, React.
    Tools: VS Code, Git, Figma.
    """
}

# --- 2. Execution ---
print("🚀 Starting AI Resume Screening Pipeline with Groq...\n")

for candidate_type, resume_text in resumes.items():
    print(f"--- Evaluating {candidate_type} ---")
    
    inputs = {
        "resume": resume_text,
        "job_description": job_description
    }
    
    try:
        # The invoke method triggers the tracing automatically
        result = pipeline.invoke(inputs)
        # Because ChatGroq returns an AIMessage object, we print the .content
        print(result.content)
    except Exception as e:
        print(f"Error processing {candidate_type}: {e}")
        
    print("\n" + "="*50 + "\n")

🚀 Starting AI Resume Screening Pipeline with Groq...

--- Evaluating Strong Candidate ---
**MATCHING SUMMARY:** The candidate's skills match the job description requirements with a few minor adjustments. The candidate has additional skills in Generative AI, which is not required but still relevant to the Data Scientist role. The candidate also has experience with Docker, which is not explicitly mentioned in the job description but is a useful tool for data scientists.

**FIT SCORE:** 92

**EXPLANATION:** The candidate meets 90% of the required skills, including Python, Machine Learning, Deep Learning, NLP, LangChain, TensorFlow, PyTorch, Jupyter, and Git. The only missing skill is not explicitly mentioned in the job description, but the candidate has additional skills that are relevant to the role. The candidate also has 5 years of experience, which meets the 4+ years requirement. The only reason for not giving a 100% fit score is that the job description does not explicitly mention Do